# 1D Geomechanics & Hydraulic Fracturing Fracability Sweet-Spotting
**Case Study:** Unconventional Horizontal Well in Vaca Muerta (2,500m to 3,000m MD)  
**Author:** Tiago Meilan  
**Discipline:** Petrophysics, Geomechanics & Multistage Completion Optimization

---

### Abstract & Problem Statement
In unconventional shale reservoirs, blind geometric stage and cluster spacing often leads to preferential fluid entry into the lowest-stress perforations (*heel-bias*), while ductile or high-stress clusters fail to initiate, stranding reserves.

This notebook demonstrates an end-to-end workflow:
1. **Log Ingestion & Acoustic Synthesis:** Parse `.las` logs and synthesize shear wave slowness ($DTS$) via the Castagna Mudrock line.
2. **Dynamic-to-Static Elastic Moduli:** Compute continuous $E_{dyn}, \nu_{dyn}$ and calibrate to static reservoir conditions ($E_{stat}, \nu_{stat}$) using the Bradford shale model.
3. **1D Mechanical Earth Model (1D MEM):** Density integration for overburden ($S_v$), Eaton's acoustic method for pore pressure ($P_p$), and Zoback's poroelastic horizontal strain model for $\sigma_{h,min}$ and $S_{Hmax}$.
4. **Composite Fracability Sweet-Spotting:** Integrate Rickman's elastic brittleness, mineralogy ($V_{sh}$), and in-situ stress confinement into a unified Fracability Index ($FI$).
5. **Engineered Completion Optimization:** Automated stage partitioning and sweet-spot cluster placement to minimize intra-stage stress variance.

## 1. Environment Setup & Engine Execution

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Ensure local src is in path
sys.path.append("..")
from src.engine import GeomechEngine
from src.petrophysics import generate_synthetic_vaca_muerta_well
from src.visualization import plot_composite_geomech_log

# 1. Ensure benchmark LAS file exists
data_dir = Path("../data/synthetic")
data_dir.mkdir(parents=True, exist_ok=True)
las_file = data_dir / "vaca_muerta_benchmark.las"

if not las_file.exists():
    print("Generating synthetic Vaca Muerta horizontal well benchmark...")
    generate_synthetic_vaca_muerta_well(output_las_path=las_file)

# 2. Initialize and configure GeomechEngine
engine = GeomechEngine(
    target_stage_length_m=60.0,
    clusters_per_stage=5,
    biot_alpha=0.85,
    tectonic_strain_x=0.00015,
    tectonic_strain_y=0.00045,
    eaton_exponent=3.0,
)

result = engine.run_from_las(las_file)
print(f"[+] Execution completed successfully for well: {result.well_data.well_name}")
print(f"    Interval: {result.well_data.start_depth:.1f}m - {result.well_data.stop_depth:.1f}m")

## 2. Quantitative Completion Audit: Geometric vs. Engineered Spacing

In [ ]:
rep = result.comparison_report

comparison_table = pd.DataFrame({
    "Metric": [
        "Mean Cluster Fracability Index (FI)",
        "Mean Intra-Stage Stress Contrast [psi]",
        "Ductile / Barrier Perforations (Risk Count)",
        "Total Perforation Clusters Placed"
    ],
    "Geometric (Blind Spacing)": [
        f"{rep.geometric_mean_fi:.3f}",
        f"{rep.geometric_mean_stress_contrast_psi:.1f} psi",
        f"{rep.geometric_ductile_clusters} clusters",
        f"{rep.total_clusters}"
    ],
    "Engineered (Sweet-Spotting)": [
        f"{rep.engineered_mean_fi:.3f} (+{rep.fi_improvement_pct}%)",
        f"{rep.engineered_mean_stress_contrast_psi:.1f} psi (-{rep.stress_contrast_reduction_pct}%)",
        f"{rep.engineered_ductile_clusters} clusters",
        f"{rep.total_clusters}"
    ]
})
comparison_table

## 3. Composite Well Log Track (Techlog / Petrel Standard)
Publication-grade 7-track continuous geomechanical well profile with inverted depth axis:

In [ ]:
fig = plot_composite_geomech_log(result, figsize=(16, 11))
plt.show()

## 4. Exporting Deliverables
Save the consolidated 1D dataset into high-speed Parquet format and export the perforation schedule for the field completion crew.

In [ ]:
out_dir = Path("../results")
out_dir.mkdir(parents=True, exist_ok=True)

result.to_parquet(out_dir / "geomech_profiles.parquet")
schedule_df = result.export_completion_schedule(out_dir / "completion_schedule.csv", design="engineered")
print(f"[+] Exported {len(schedule_df)} perforation clusters to CSV")
schedule_df.head(10)